In [10]:
from IPython.display import HTML
HTML("""
<style>
div.output_area pre,
div.output_subarea pre,
div.output_text pre {
    white-space: pre-wrap !important;
    word-wrap: break-word !important;
    max-width: 900px !important;
}
</style>
""")
import os
import time
import pandas as pd
from semantic_search import SemanticSearcher
from BM_25_search import BM25Searcher
from tqdm import tqdm

# Specify paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
REMARKS_PATH = os.path.join(PROJECT_ROOT, "data/processed/listing_remarks.csv")
EMB_PATH = os.path.join(PROJECT_ROOT, "data/processed/remarks_embeddings.npy")

# Instantiate both searchers
searcher = SemanticSearcher(remarks_path=REMARKS_PATH, emb_path=EMB_PATH)
bm25 = BM25Searcher(remarks_path=REMARKS_PATH)

# Load semantic index (never rebuilds here)
if not os.path.exists(searcher.emb_path):
    raise FileNotFoundError(
        f"Embeddings not found at {searcher.emb_path}. "
        "Run the embedding builder script before using this notebook."
    )
searcher.load_index()

# Semantic model warmup
t0 = time.perf_counter()
_ = searcher.model.encode(["warmup"], convert_to_numpy=True)
t1 = time.perf_counter()
print(f"Warmup time: {(t1 - t0) * 1000:.2f} ms")

# Single-query latency test
query = "condo with granite countertops and pool"
t2 = time.perf_counter()
results = searcher.search(query, top_k=10)
t3 = time.perf_counter()
print(f"Post-warmup query latency: {(t3 - t2) * 1000:.2f} ms")

# Full user-experienced latency w/ semantic search
start = time.perf_counter()
with tqdm(total=1, bar_format="{l_bar}{bar}| {elapsed}") as pbar:
    results = searcher.search(query, top_k=10)
    pbar.update(1)
end = time.perf_counter()
print(f"Full semantic query latency: {(end - start) * 1000:.2f} ms")

# Comparison queries
df = pd.read_csv(REMARKS_PATH)
listings = df["remarks"].fillna("").tolist()

queries = [
    "condo with granite countertops and pool",
    "house near good schools with a big backyard",
    "modern townhouse with open floor plan",
    "luxury home with mountain views",
    "starter home under 400k"
]

results = []

for q in queries:
    # BM25 timing
    t0 = time.perf_counter()
    bm25_res = bm25.search(q, top_k=10)
    t1 = time.perf_counter()

    # Semantic timing
    t2 = time.perf_counter()
    sem_res = searcher.search(q, top_k=10)
    t3 = time.perf_counter()

    results.append({
        "query": q,
        "bm25_latency_ms": (t1 - t0) * 1000,
        "semantic_latency_ms": (t3 - t2) * 1000,
        "bm25_top_result": bm25_res[0][0],
        "semantic_top_result": sem_res[0][0]
    })

comparison_df = pd.DataFrame(results)
comparison_df

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5950.02it/s]


Loaded cached embeddings: (1000, 384)
Loaded 1000 listing remarks
FAISS index rebuilt from cached data
Warmup time: 16.24 ms
Post-warmup query latency: 8.36 ms


100%|██████████| 00:00

Full semantic query latency: 9.52 ms


,query,bm25_latency_ms,semantic_latency_ms,bm25_top_result,semantic_top_result
0,condo with granite countertops and pool,2.327584,12.342750,This beautifully maintained second-floor unit ...,Welcome to this stunning tri-level townhome-st...
1,house near good schools with a big backyard,1.436292,10.808958,"Two houses, Main house is 3 bedroom 1 bath, go...","NEW LISTING!!**OPEN HOME SAT & SUN, 1-31 & 2-1..."
2,modern townhouse with open floor plan,1.132792,7.915583,One of rare locations in Princeton courts town...,Welcome to this townhouse-style home offering ...
3,luxury home with mountain views,0.809583,7.245541,"Welcome to NOLA at Escena, a sleek new communi...",Breathtaking panoramic views and upgrades surp...
4,starter home under 400k,0.628500,6.525750,"Bright and spacious 3-bedroom, 2.5-bath home f...",16.64 very usable fenced acres. Mini ranch or ...


A strict comparison of latency in ms does not give us real insight into the usefulness of BM25 vs our semantic searcher. A human user will not experience a marked difference between 1 and 399 ms. Where I expect a semantic searcher to really improve a user's experience is in the flexibility and relevance of search results. I compare those more fully below.

In [15]:
def show_result(text, score):
    html = f"""
    <div style='max-width:700px; white-space:normal; line-height:1.4; margin-bottom:1em;'>
        <b>Score:</b> {score:.4f}<br>
        {text}
    </div>
    """
    display(HTML(html))

for q in queries:
    print("\n" + "="*80)
    print(f"QUERY: {q}\n")

    bm25_res = bm25.search(q, top_k=5)
    sem_res = searcher.search(q, top_k=5)

    print("BM25 RESULTS:")
    for text, score in bm25_res:
        show_result(text, score)

    print("SEMANTIC RESULTS:")
    for text, score in sem_res:
        show_result(text, score)


QUERY: condo with granite countertops and pool

BM25 RESULTS:


SEMANTIC RESULTS:



QUERY: house near good schools with a big backyard

BM25 RESULTS:


SEMANTIC RESULTS:



QUERY: modern townhouse with open floor plan

BM25 RESULTS:


SEMANTIC RESULTS:



QUERY: luxury home with mountain views

BM25 RESULTS:


SEMANTIC RESULTS:



QUERY: starter home under 400k

BM25 RESULTS:


SEMANTIC RESULTS:
